# Apache Polaris — Reset vs. Rotate: Credential Invalidation Timing

## Concept
A focused, live-verifiable test of ONE specific question: **when exactly
does a superseded secret stop working** — after `/reset` (root-driven) vs.
after `/rotate` (self-service)? Confirmed live (2026-07-08) that they are
NOT the same:

| API                              | Caller             | Secret slots affected            | Previous secret dies after |
| --------------------------------- | ------------------ | --------------------------------- | --------------------------- |
| `POST /principals/{name}/reset`   | root / admin        | internally rotates TWICE per call | **1 call**                  |
| `POST /principals/{name}/rotate`  | the principal itself | rotates ONCE per call            | **2 calls**                 |

Polaris stores a **main** and a **secondary** secret per principal
(`PolarisPrincipalSecrets.matchesSecret` accepts either hash). A single
`/rotate` only demotes the current main secret to secondary — the demoted
secret **keeps authenticating** for exactly one more generation. `/reset`
performs two of these rotations internally in one HTTP call, so it fully
evicts whatever was active immediately before it — no grace period.

This is NOT about role wiring (principal-role/catalog-role survival is
already covered live by `polaris_rotate_principal_test.ipynb`) — this
notebook exists solely to pin down the invalidation-timing difference with
explicit assertions, since it's easy to get wrong (we did, initially — see
`MEMORY.md`'s 2026-07-08 entry and the corrected assertions in the two
sibling notebooks).

## Test plan
```
1. Provision   a throwaway test principal (+ principal-role), capture its
               INITIAL credentials from the create_principal response
2. SCENARIO A  — ROTATE (self-service), two calls:
     rotate #1: assert the pre-rotate secret STILL works (now secondary)
     rotate #2: assert the pre-rotate secret is NOW dead (evicted)
3. SCENARIO B  — RESET (root), one call:
     reset #1:  assert the pre-reset secret is dead IMMEDIATELY (1 call)
4. Bonus       — reset again on a fresh generation, to confirm this isn't
               tied to the initial creation-time secret specifically
5. Summary     side-by-side printout of the two eviction timings
6. Teardown    delete principal-role, principal
7. Sweep       backstop for any `reset-vs-rotate-*` leftovers
```

## Safety — LOCAL DEVICE ONLY
Same hardcoded-local convention as the rest of `admin/`. Creates and deletes
a throwaway principal only; no catalog is provisioned (out of scope here).

## How to run
Restart & Run All. No parameters to set.


In [1]:
# ============================================================
# Prerequisites:  uv add requests
# ============================================================
import subprocess, sys, os

REQUIRED = ['requests']

def install(pkg):
    result = subprocess.run(['uv', 'add', pkg], capture_output=True, text=True, cwd=os.getcwd())
    if result.returncode == 0:
        return '✅', 'uv'
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg], capture_output=True, text=True)
    return ('✅', 'pip') if result.returncode == 0 else ('❌', 'failed')

for pkg in REQUIRED:
    status, method = install(pkg)
    print(f'  {status} {pkg} ({method})')
print('\n⚠️  Restart kernel after first install.')

  ✅ requests (uv)

⚠️  Restart kernel after first install.


In [2]:
# ============================================================
# Setup — hardcoded local device only (see Safety note above)
# ============================================================
import sys, pathlib, time

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
from polaris_rest import PolarisREST

POLARIS_URL = 'http://192.168.139.2:8181'
REALM       = 'POLARIS'
ROOT        = 'root'
ROOT_SECRET = 'polaris-secret'

assert any(t in POLARIS_URL for t in ("127.0.0.1", "localhost", "192.168.139.2")), \
    f"reset_vs_rotate_test is LOCAL ONLY — refusing non-local host: {POLARIS_URL}"

pc = PolarisREST(POLARIS_URL, REALM)
r = pc.get_token(ROOT, ROOT_SECRET)
pc.token = r.json()['access_token']
print('✅ Connected as root')

✅ Connected as root


In [3]:
# ============================================================
# Parameters — auto-generated, unique per run
# ============================================================
TEST_PREFIX    = 'reset-vs-rotate-'
TEST_PRINCIPAL = f'{TEST_PREFIX}{int(time.time())}'
TEST_ROLE      = f'{TEST_PRINCIPAL}_role'

print(f'TEST_PRINCIPAL = {TEST_PRINCIPAL}')
print(f'TEST_ROLE      = {TEST_ROLE}')

TEST_PRINCIPAL = reset-vs-rotate-1783475933
TEST_ROLE      = reset-vs-rotate-1783475933_role


In [4]:
# ============================================================
# Step 1 — Provision a throwaway principal, capture its INITIAL credentials
# ============================================================
r = pc.create_principal(TEST_PRINCIPAL)
assert r.status_code in (200, 201), f'create_principal failed: {r.status_code} {r.text}'
body = r.json()
INIT_CID  = body.get('principal', {}).get('clientId')
INIT_SECRET = body.get('credentials', {}).get('clientSecret')
assert INIT_CID and INIT_SECRET, f'create_principal response missing credentials: {body}'
print(f'✅ principal created: {TEST_PRINCIPAL}')
print(f'   initial clientId:     {INIT_CID}')
print(f'   initial clientSecret: {INIT_SECRET}')

r = pc.create_principal_role(TEST_ROLE)
assert r.status_code in (200, 201), f'create_principal_role failed: {r.status_code} {r.text}'
r = pc.assign_principal_role_to_principal(TEST_PRINCIPAL, TEST_ROLE)
assert r.status_code in (200, 201), f'assign failed: {r.status_code} {r.text}'
print(f'✅ {TEST_ROLE} assigned to {TEST_PRINCIPAL}')

# sanity check: the initial credentials actually work before we start
r = pc.get_token(INIT_CID, INIT_SECRET, scope=f'PRINCIPAL_ROLE:{TEST_ROLE}')
assert r.status_code == 200, f'Initial credentials do not authenticate: {r.status_code} {r.text}'
print('✅ initial credentials authenticate — baseline confirmed')

✅ principal created: reset-vs-rotate-1783475933
   initial clientId:     a422d7e5acb20068
   initial clientSecret: 0bc22e6f87a0127fb49363a45aaa1589
✅ reset-vs-rotate-1783475933_role assigned to reset-vs-rotate-1783475933
✅ initial credentials authenticate — baseline confirmed


In [5]:
# ============================================================
# Step 2 — SCENARIO A: ROTATE (self-service) — two calls to fully evict
# ============================================================
def as_principal(cid, secret):
    r = pc.get_token(cid, secret, scope=f'PRINCIPAL_ROLE:{TEST_ROLE}')
    assert r.status_code == 200, f'auth failed for {cid}: {r.status_code} {r.text}'
    return r.json()['access_token']

# --- rotate #1 ---
tok0 = as_principal(INIT_CID, INIT_SECRET)
r = pc.rotate_credentials(TEST_PRINCIPAL, token=tok0)
assert r.status_code == 200, f'rotate #1 failed: {r.status_code} {r.text}'
r1 = r.json()['credentials']
R1_CID, R1_SECRET = r1['clientId'], r1['clientSecret']
print(f'✅ rotate #1 → new main: {R1_CID}')

r_init_after_1 = pc.get_token(INIT_CID, INIT_SECRET, scope=f'PRINCIPAL_ROLE:{TEST_ROLE}')
assert r_init_after_1.status_code == 200, (
    f'Expected INITIAL creds to still work after ONE rotate (secondary slot) — '
    f'got {r_init_after_1.status_code}.'
)
print('✅ AFTER 1 rotate: initial creds STILL work (now the secondary secret)')

r_r1_check = pc.get_token(R1_CID, R1_SECRET, scope=f'PRINCIPAL_ROLE:{TEST_ROLE}')
assert r_r1_check.status_code == 200
print('✅ AFTER 1 rotate: new (main) creds work')

# --- rotate #2 ---
tok1 = as_principal(R1_CID, R1_SECRET)
r = pc.rotate_credentials(TEST_PRINCIPAL, token=tok1)
assert r.status_code == 200, f'rotate #2 failed: {r.status_code} {r.text}'
r2 = r.json()['credentials']
R2_CID, R2_SECRET = r2['clientId'], r2['clientSecret']
print(f'✅ rotate #2 → new main: {R2_CID}')

r_init_after_2 = pc.get_token(INIT_CID, INIT_SECRET, scope=f'PRINCIPAL_ROLE:{TEST_ROLE}')
assert r_init_after_2.status_code != 200, (
    f'Expected INITIAL creds to be FULLY EVICTED after a second rotate — '
    f'still got {r_init_after_2.status_code}.'
)
print(f'✅ AFTER 2 rotates: initial creds are FINALLY dead ({r_init_after_2.status_code})')

r_r1_after_2 = pc.get_token(R1_CID, R1_SECRET, scope=f'PRINCIPAL_ROLE:{TEST_ROLE}')
assert r_r1_after_2.status_code == 200
print('✅ AFTER 2 rotates: rotate-#1 creds still work (now the secondary secret)')

print('\n✅ CONFIRMED: /rotate takes exactly 2 calls to fully evict a given secret.')

✅ rotate #1 → new main: a422d7e5acb20068
✅ AFTER 1 rotate: initial creds STILL work (now the secondary secret)
✅ AFTER 1 rotate: new (main) creds work
✅ rotate #2 → new main: a422d7e5acb20068
✅ AFTER 2 rotates: initial creds are FINALLY dead (401)
✅ AFTER 2 rotates: rotate-#1 creds still work (now the secondary secret)

✅ CONFIRMED: /rotate takes exactly 2 calls to fully evict a given secret.


In [6]:
# ============================================================
# Step 3 — SCENARIO B: RESET (root) — ONE call fully evicts
# ============================================================
# Baseline: R2 (from the end of Scenario A) is the current valid pair.
r = pc.get_token(R2_CID, R2_SECRET, scope=f'PRINCIPAL_ROLE:{TEST_ROLE}')
assert r.status_code == 200, 'baseline for reset scenario is not valid — cannot proceed'
print(f'✅ baseline confirmed: pre-reset creds ({R2_CID}) currently authenticate')

r = pc.reset_principal_credentials(TEST_PRINCIPAL)
assert r.status_code == 200, f'reset failed: {r.status_code} {r.text}'
reset1 = r.json()['credentials']
RESET1_CID, RESET1_SECRET = reset1['clientId'], reset1['clientSecret']
print(f'✅ reset → new main: {RESET1_CID}')

r_pre_reset_after = pc.get_token(R2_CID, R2_SECRET, scope=f'PRINCIPAL_ROLE:{TEST_ROLE}')
assert r_pre_reset_after.status_code != 200, (
    f'Expected the pre-reset secret to be dead after just ONE reset call — '
    f'still got {r_pre_reset_after.status_code}.'
)
print(f'✅ AFTER 1 reset: pre-reset creds are IMMEDIATELY dead ({r_pre_reset_after.status_code})')

# rotate-#1 creds (R1) were already evicted in Scenario A and should stay dead
r_r1_still = pc.get_token(R1_CID, R1_SECRET, scope=f'PRINCIPAL_ROLE:{TEST_ROLE}')
assert r_r1_still.status_code != 200
print('✅ (sanity) long-evicted rotate-#1 creds remain dead')

r_reset1_check = pc.get_token(RESET1_CID, RESET1_SECRET, scope=f'PRINCIPAL_ROLE:{TEST_ROLE}')
assert r_reset1_check.status_code == 200
print('✅ new reset creds authenticate')

print('\n✅ CONFIRMED: /reset takes exactly 1 call to fully evict a given secret —')
print('   no secondary-secret grace period, unlike /rotate.')

✅ baseline confirmed: pre-reset creds (a422d7e5acb20068) currently authenticate
✅ reset → new main: a422d7e5acb20068
✅ AFTER 1 reset: pre-reset creds are IMMEDIATELY dead (401)
✅ (sanity) long-evicted rotate-#1 creds remain dead
✅ new reset creds authenticate

✅ CONFIRMED: /reset takes exactly 1 call to fully evict a given secret —
   no secondary-secret grace period, unlike /rotate.


In [7]:
# ============================================================
# Step 4 — Bonus: confirm this isn't specific to the very first secret
# ============================================================
# Reset again, starting from RESET1 (not the original creation-time secret),
# to prove single-call full eviction is RESET's general behavior at any
# generation, not a fluke of how the principal was created.
r = pc.reset_principal_credentials(TEST_PRINCIPAL)
assert r.status_code == 200, f'second reset failed: {r.status_code} {r.text}'
reset2 = r.json()['credentials']
RESET2_CID, RESET2_SECRET = reset2['clientId'], reset2['clientSecret']

r_reset1_after = pc.get_token(RESET1_CID, RESET1_SECRET, scope=f'PRINCIPAL_ROLE:{TEST_ROLE}')
assert r_reset1_after.status_code != 200, (
    f'Expected RESET1 creds dead after a second reset — got {r_reset1_after.status_code}.'
)
print(f'✅ RESET1 creds dead immediately after the next reset too ({r_reset1_after.status_code})')

r_reset2_check = pc.get_token(RESET2_CID, RESET2_SECRET, scope=f'PRINCIPAL_ROLE:{TEST_ROLE}')
assert r_reset2_check.status_code == 200
print('✅ newest reset creds authenticate — /reset\'s 1-call eviction is consistent across generations.')

✅ RESET1 creds dead immediately after the next reset too (401)
✅ newest reset creds authenticate — /reset's 1-call eviction is consistent across generations.


In [8]:
# ============================================================
# Step 5 — Summary
# ============================================================
print('=' * 64)
print(f'{"API":<10} {"caller":<20} {"calls to fully evict a secret":<30}')
print('-' * 64)
print(f'{"/rotate":<10} {"principal (self)":<20} {"2":<30}')
print(f'{"/reset":<10} {"root/admin":<20} {"1":<30}')
print('=' * 64)
print('\nBoth ultimately achieve the same end state (old secret unusable) —')
print('they just differ in how many calls it takes to get there, and that')
print('matters for anything that assumes immediate invalidation.')

API        caller               calls to fully evict a secret 
----------------------------------------------------------------
/rotate    principal (self)     2                             
/reset     root/admin           1                             

Both ultimately achieve the same end state (old secret unusable) —
they just differ in how many calls it takes to get there, and that
matters for anything that assumes immediate invalidation.


In [9]:
# ============================================================
# Step 6 — Teardown (idempotent — safe to re-run)
# ============================================================
r = pc.delete_principal_role(TEST_ROLE)
print(f"{'✅' if r.status_code in (200,204,404) else '⚠️ '} [{r.status_code}] principal-role: {TEST_ROLE}")

r = pc.delete_principal(TEST_PRINCIPAL)
print(f"{'✅' if r.status_code in (200,204,404) else '⚠️ '} [{r.status_code}] principal: {TEST_PRINCIPAL}")

assert pc.get_principal(TEST_PRINCIPAL).status_code == 404
assert pc.get_principal_role(TEST_ROLE).status_code == 404
print('\n✅ Teardown verified.')

✅ [204] principal-role: reset-vs-rotate-1783475933_role
✅ [204] principal: reset-vs-rotate-1783475933

✅ Teardown verified.


In [10]:
# ============================================================
# Step 7 — Safety sweep (backstop for a previous crashed run)
# ============================================================
r = pc.list_principals()
leftover = [p['name'] for p in r.json().get('principals', []) if p['name'].startswith(TEST_PREFIX)]
for name in leftover:
    pc.delete_principal(name)
    pc.delete_principal_role(f'{name}_role')
    print(f'🧹 swept leftover: {name}')

if not leftover:
    print('✅ nothing to sweep.')

✅ nothing to sweep.
